# Cost Management & Budget Optimization Guide

**Track, forecast, and optimize cloud costs while maintaining performance and reliability.**

- Cost tracking & attribution
- Budget planning & forecasting
- Resource optimization strategies
- Right-sizing & consolidation
- Waste identification & reduction
- Reserved instance strategies
- Cost anomaly detection
- Reporting & visibility

**Estimated time:** 50 minutes | **Skill level:** Intermediate-Advanced | **For:** DevOps, platform engineers, finance liaisons


## 1. Cloud Cost Structure

### Azure Pricing Model

```
Compute:
├─ VMs (pay-per-second)
│  ├─ Standard (pay-as-you-go)
│  ├─ Reserved Instances (1-3 year commitment)
│  └─ Spot instances (up to 90% discount)
│
Storage:
├─ Blob storage (per GB, access tier dependent)
├─ SQL Database (DTU or vCore based)
├─ Cosmos DB (RU/s provisioned)
└─ Data transfer (egress costs)

Services:
├─ Azure Functions (execution time + memory)
├─ Azure Cognitive Services (requests/month)
├─ Application Insights (data ingestion/retention)
└─ Service Bus (messages)
```

### Cost Attribution Model

```markdown
| Service          | Monthly Cost | Driver         | Owner        |
| ---------------- | ------------ | -------------- | ------------ |
| Web API          | $1,200       | 3x Standard D4 | Backend team |
| Training cluster | $2,500       | GPU hours      | ML team      |
| Database         | $800         | DTU hours      | DevOps       |
| Storage          | $150         | 500 GB blob    | All teams    |
| Networking       | $200         | Data transfer  | DevOps       |
| Monitoring       | $400         | Data ingestion | Ops          |

Total Monthly: $5,250
```

### Cost Distribution

```
Fixed Costs (70%):
├─ Development environment: $500/month
├─ Production infrastructure: $2,000/month
├─ Database (reserved): $800/month
└─ Monitoring: $400/month

Variable Costs (30%):
├─ Training (GPU): $600-1,000/month
├─ Data transfer: $150-300/month
├─ Storage growth: $50-100/month
└─ Testing/CI: $200-300/month
```


## 2. Budget Planning & Forecasting

### Monthly Budget Template

```yaml
# budget_2024_q1.yaml

monthly_budget: $5,500

allocations:
    production:
        compute: $2,000 # 3x D4 Standard
        database: $800 # SQL Always-On
        storage: $100 # Blob + archive
        total: $2,900

    training:
        compute: $800 # 20 GPU hours/month
        storage: $50
        total: $850

    development:
        compute: $500 # Dev machines
        database: $150 # Shared dev DB
        total: $650

    infrastructure:
        networking: $200
        monitoring: $400
        backup: $100
        total: $700

contingency: 5% # $275

tolerance:
    normal: 0-5%
    alert: 5-10%
    escalate: >10
```

### Cost Forecasting

```markdown
Historical Trend Analysis:

- Q4 2024: $5,200 (baseline)
- Q1 2025: $5,400 (6% increase expected due to new features)
- Q2 2025: $5,800 (expected new customer = +$400)

Forecast Formula:
Expected = Baseline + Growth Factor + Seasonal Factor

Example:

- Baseline: $5,200
- Growth (10% YoY): +$520
- Seasonal (Q2 peak): +$100
- Expected Q2: $5,820

Accuracy: ±10% is typical
```

### Forecasting Factors

```markdown
| Factor             | Impact | Trend       |
| ------------------ | ------ | ----------- |
| User growth        | +2-3%  | Predictable |
| Training volume    | +5-10% | Variable    |
| Data volume        | +1-2%  | Steady      |
| Feature releases   | +2-5%  | Seasonal    |
| Performance tuning | -1-3%  | Ongoing     |
```


## 3. Resource Optimization Strategies

### Virtual Machine Sizing

```markdown
Current Setup (Expensive):
├─ Production: 3x D4 Standard ($2,000/month)
│ ├─ CPU usage: ~20% average
│ ├─ Memory: ~40% average
│ └─ Overkilled ❌
│
└─ Problem: Right-sized for peak (2% of time)

Optimization:
├─ Switch to: 2x D3 Standard + autoscaling
├─ Cost: $1,300/month
├─ Savings: $700/month (35% reduction)
├─ Process: Monitor 2 weeks before committing
│
├─ Timeline:
│ - Week 1: Deploy parallel
│ - Week 2: Route traffic (10% → 50% → 100%)
│ - Week 3: Monitor and tune
│ - Week 4: Decommission old (if stable)
│
└─ Risk: Low (can rollback in minutes)
```

### Reserved Instance Strategy

```
Decision Tree:
├─ Is workload stable? (same 24/7/365)
│  ├─ YES → Buy 1-year RI (save 25-35%)
│  └─ NO → Go to next question
│
├─ Is workload predictable? (consistent patterns)
│  ├─ YES → Buy 1-year RI for base + on-demand for spikes
│  └─ NO → Go to next question
│
└─ Use on-demand or Spot (with fallback)

Example Calculation:
Machine: D4 Standard
- On-demand: $200/month × 12 = $2,400/year
- 1-year RI: $1,700/year
- Savings: $700/year (29%)
- Payback: 3 months

Our Application:
Production baseline: Always 3 VMs running
├─ Reserve: 3x D4 × 12 months
├─ Cost: $5,100 (vs $6,000 on-demand)
├─ Savings: $900/year
└─ Risk: Low (can pay on-demand if spike needed)
```

### Database Optimization

```markdown
SQL Database Sizing:

Current (Expensive):
├─ DTU-based: 200 DTU ($150/month)
├─ Utilization: 15-20% average
├─ Peak: 40% (3-4 times/day)
└─ Problem: Over-provisioned

Analysis:
├─ 95th percentile usage: 60 DTU
├─ Could downsize to: 100 DTU tier
├─ Cost reduction: $50/month
├─ Risk: High if spike happens (queries slow down)

Strategy:
├─ Downsize to 100 DTU in test environment
├─ Monitor for 2 weeks
├─ Watch during peak load times
├─ Add alerting for DTU > 80%
├─ Keep option to scale up (5 min)

Cosmos DB (No-SQL):
├─ Current: 400 RU/s provisioned
├─ Average: 80 RU/s used
├─ Cost: $200/month

Optimization:
├─ Use autoscale (100-400 RU/s range)
├─ Cost: 1.25 × on-demand = $125/month
├─ Savings: $75/month
└─ Benefit: Automatic scaling, no monitoring needed
```

### Storage Optimization

```markdown
Current:
├─ Hot tier: 200 GB @ $0.0184/GB = $3.68/month
├─ Warm tier: 300 GB @ $0.0092/GB = $2.76/month
├─ Archive: 1,000 GB @ $0.00099/GB = $0.99/month
└─ Total: $7.43/month

Problem: Access patterns:
├─ Hot: Recent data (used daily)
├─ Warm: Data from last 90 days
├─ Archive: Historical (rarely accessed)

But we're storing too much in Hot!

Optimization:
├─ Inventory: 200 GB in hot could move to cool
├─ Move: 100 GB unused → archive
├─ Result:
│ ├─ Hot: 100 GB
│ ├─ Warm: 300 GB
│ └─ Archive: 1,100 GB
│ └─ New cost: $2.75/month
│ └─ Savings: $4.68/month (63% reduction)

Implementation:
├─ Use lifecycle policies to auto-archive
├─ No manual work after setup
├─ Test restore from archive first
```


## 4. Waste Identification & Reduction

### Common Cost Drains

| Issue                      | Impact       | Fix                     | Savings |
| -------------------------- | ------------ | ----------------------- | ------- |
| Forgotten dev environments | $200-500/mo  | Stop/delete unused      | $300    |
| Oversized databases        | $100-300/mo  | Right-size to 80% peak  | $150    |
| Excess data transfer       | $50-200/mo   | Optimize queries, cache | $100    |
| Unoptimized backups        | $50-100/mo   | Retention policies      | $75     |
| Underutilized GPU          | $500-1000/mo | Batch jobs better       | $400    |

### Waste Detection Process

```markdown
Step 1: Visibility (Monthly)
├─ Export cost data from Azure
├─ Tag all resources by team/purpose
├─ Group costs by service
└─ Identify top 5 cost centers

Step 2: Analysis (Weekly)
├─ Low utilization resources (< 20% usage)
├─ Idle resources (not accessed)
├─ Over-provisioned instances
└─ Wasted licenses

Step 3: Action (Ongoing)
├─ Document findings
├─ Propose optimization
├─ Get approval
├─ Implement safely (test first)
└─ Monitor impact

Step 4: Prevention
├─ Set up alerts
├─ Tagging requirements
├─ Review before provisioning
└─ Shutdown unused resources monthly
```

### Detection Tools

```python
# Script to find unused resources
import datetime
from azure.monitor.query import MetricsQueryClient

def find_underutilized_vms():
    """Find VMs using < 20% CPU for last 30 days"""
    metrics_client = MetricsQueryClient()

    vms = [
        'prod-web-01',
        'prod-web-02',
        'dev-machine',
    ]

    for vm in vms:
        result = metrics_client.query_resource(
            resource_id=f"/subscriptions/{sub_id}/resourceGroups/"
                       f"rg-prod/providers/Microsoft.Compute/virtualMachines/{vm}",
            metric_names=['Percentage CPU'],
            timespan=datetime.timedelta(days=30),
            granularity=datetime.timedelta(hours=1),
            aggregations=['average']
        )

        avg_cpu = result.metrics['Percentage CPU'].timeseries[0].data[0].average

        if avg_cpu < 20:
            print(f"{vm}: {avg_cpu:.1f}% CPU - Consider downsizing")
        else:
            print(f"{vm}: {avg_cpu:.1f}% CPU - OK")
```


## 5. Cost Anomaly Detection

### Setting Up Alerts

```yaml
# cost-alerts.yaml

alert:
  name: "Monthly cost exceeds budget"
  type: "absolute"
  threshold: $5,750  # 5% over budget
  frequency: "daily"
  notification:
    - email: devops-team@company.com
    - slack: "#cost-alerts"

alert:
  name: "Category surge detected"
  type: "threshold"
  scope: "compute"
  change: "+20% from baseline"
  frequency: "daily"
  notification:
    - email: devops-team@company.com
    - slack: "#cost-alerts"

alert:
  name: "Resource idle"
  type: "utilization"
  resource: "vm"
  condition: "cpu < 5% AND network < 1 Mbps"
  duration: "7 days"
  frequency: "weekly"
  notification:
    - slack: "#ops"
```

### Anomaly Response Workflow

```markdown
Step 1: Detection (Automated)
└─ Alert fires when cost jumps 20%+

Step 2: Investigation (30 minutes)
├─ Check what changed:
│ ├─ New resources created?
│ ├─ Scaling event?
│ ├─ New job running?
│ └─ Billing error?
└─ Run cost breakdown report

Step 3: Diagnosis (1 hour)
├─ Confirm legitimate use (yes/no)
├─ If legitimate: forecast impact
├─ If waste: identify root cause
└─ Document findings

Step 4: Action (same day if urgent)
├─ If legitimate:
│ ├─ Update budget forecast
│ ├─ Set new baseline
│ └─ Monitor for stabilization
│
└─ If waste:
├─ Stop/delete resource
├─ Implement fix
├─ Prevent recurrence
└─ Update playbook

Step 5: Communication
├─ Slack update to team
├─ Root cause analysis (if major)
├─ Prevention steps (if needed)
```

### Example Anomaly

```markdown
Alert: Compute costs up 45% ($950 vs $500 baseline)

Investigation:
├─ Check Azure portal
├─ Last 24 hours: 2 new D8 VMs spun up
├─ Created: Training team (unscheduled job)
├─ Duration: Still running
├─ Cost: $800 for 24 hours

Diagnosis:
├─ Legitimate: Training needs GPU?
│ └─ Call to verify scope/duration
├─ "Oops, forgot to stop it!"
│ └─ Stop immediately, reduce cost to $50

Action:
├─ Stop the VMs
├─ Set autoshutdown for future
├─ Train team on cost controls
├─ Add approval flow for large VMs

Result:
├─ Cost recovered: -$750
├─ Prevention: Autoshutdown policy
└─ Learning: Document in playbook
```


## 6. Cost Reporting & Visibility

### Monthly Cost Report

```markdown
# Monthly Cost Report - December 2024

## Executive Summary

- Total cost: $5,420 (2.5% over budget of $5,280)
- YoY growth: +12% (expected)
- Monthly trend: Increasing steadily

## Cost Breakdown

### By Service

| Service    | Cost   | MoM  | Target |
| ---------- | ------ | ---- | ------ |
| Compute    | $2,900 | +3%  | $2,800 |
| Database   | $800   | 0%   | $800   |
| Storage    | $150   | -5%  | $150   |
| Networking | $220   | +10% | $200   |
| Monitoring | $350   | 0%   | $400   |

### By Team

| Team     | Cost   | Utilization | Owner |
| -------- | ------ | ----------- | ----- |
| Backend  | $2,200 | 95%         | Alice |
| ML       | $1,500 | 80%         | Bob   |
| Frontend | $800   | 90%         | Carol |
| DevOps   | $920   | 85%         | Dave  |

## Optimization Opportunities

1. ✅ Database downsize: Save $50/month
    - Status: Approved, starting next week
    - Owner: DevOps team

2. 🔄 VM consolidation: Save $200/month
    - Status: In testing, 2 weeks remaining
    - Owner: Backend team

3. ⏳ Storage tiering: Save $30/month
    - Status: Planned for Q1
    - Owner: DevOps team

## Key Metrics

- Cost per transaction: $0.000043 (improved)
- Cost per user: $0.15 (stable)
- Cost per compute unit: $1.20 (improved)

## Next Month Focus

- Complete VM consolidation
- Implement autoscale policies
- Training on cost awareness
```

### Cost Dashboard

```markdown
Real-time Cost Dashboard:

Daily Spend:
This month: $187 (average)
Yesterday: $142
Today: $190 (projected daily)

Budget Status:
Spent: $5,420 (102.6%)
Remaining: $-140
Forecast: $5,610 (5.6% over)

Top Services:

1. Compute: $2,900 (53%)
2. Database: $800 (15%)
3. Networking: $220 (4%)
4. Monitoring: $350 (6%)
5. Storage: $150 (3%)

Alerts:
🔴 Over budget
⚠️ Networking costs up 10%
✅ Storage trending down

Trends (6 months):
Jun: $4,800 | Jul: $4,950 | Aug: $5,100
Sep: $5,280 | Oct: $5,350 | Nov: $5,420
```


## 7. Reserved Instances & Commitment Planning

### Reservation Strategy

```
Commitment Levels:

Pay-as-you-go:
├─ Flexibility: 100% (can stop anytime)
├─ Cost: Baseline (100%)
└─ Use for: Development, spikes

1-Year RI:
├─ Discount: 25-35% savings
├─ Commitment: Non-refundable
├─ Use for: Stable production base

3-Year RI:
├─ Discount: 40-50% savings
├─ Commitment: Non-refundable
├─ Use for: Core infrastructure

Spot Instances:
├─ Discount: 60-90% savings
├─ Reliability: Up to 90% cheaper
├─ Use for: Interruptible (training, batch jobs)
└─ Risk: Can be reclaimed with 30s notice
```

### Reservation Calculator

```markdown
Machine: D4 Standard
Availability: 24/7/365 (continuous)

On-Demand Cost:
├─ Monthly: $200
├─ Annual: $2,400
└─ 3-year: $7,200

1-Year RI Cost:
├─ Upfront: $1,600
├─ On-demand for overages: $100/month × 0 months
├─ Total: $1,600
└─ Savings: $800/year (33%)

3-Year RI Cost:
├─ Upfront: $4,800
├─ On-demand for overages: $100/month × 0 months
├─ Total: $4,800
└─ Savings: $2,400/3yr = $800/year (33%)

Our Production:
├─ Reserved instances: 3x D4 (1-year)
├─ Cost: $4,800/year
├─ vs on-demand: $7,200/year
├─ Savings: $2,400/year (33%)
└─ Still flexible: Can purchase on-demand if spike
```

### Commitment Management

```yaml
# reservations.yaml
reservations:
    d4_compute:
        resource: "Virtual Machine - D4 Standard"
        quantity: 3
        term: "1 Year"
        start: "2025-01-01"
        end: "2025-12-31"
        cost_annual: $4,800
        savings_vs_ondemand: $2,400
        owner: "devops-team"

    sql_database:
        resource: "SQL Database - S4"
        quantity: 1
        term: "1 Year"
        start: "2025-01-01"
        end: "2025-12-31"
        cost_annual: $800
        savings_vs_ondemand: $300
        owner: "devops-team"

upcoming_expirations:
    - date: "2025-12-31"
      resource: "d4_compute"
      action: "Renew or evaluate alternatives"
      owner: "devops-team"
      deadline: "2025-11-30"
```


## 8. Cost Governance & Controls

### Tagging Strategy

```yaml
# Every resource must have these tags:

Environment:
    - production
    - staging
    - development
    - sandbox

Team:
    - backend
    - frontend
    - ml
    - devops

CostCenter:
    - engineering
    - product
    - sales
    - marketing

Project:
    - aria-platform
    - research
    - infrastructure

Owner:
    - alice@company.com
    - bob@company.com
    # (actual email of person responsible)
```

### Cost Control Policies

```markdown
Policy: VM Size Restrictions

In Production:
✅ Allowed: D2, D3, D4 (tested, approved)
❌ Blocked: D5+, E-series, F-series (auto-denied)

In Staging:
✅ Allowed: D2, D3 (cost control)
❌ Blocked: Larger than D3

In Development:
✅ Allowed: All sizes
⚠️ Alert: If D4+ for > 24 hours
⚠️ Auto-stop: After 8 hours inactive

Implementation:
├─ Azure Policy (enforce via RBAC)
├─ Approval workflow (for exceptions)
└─ Alerts (usage above threshold)
```

### Spending Limits

```markdown
Departmental Budgets:

Engineering: $2,500/month
├─ Compute: $1,500 (60%)
├─ Database: $600 (24%)
├─ Development: $400 (16%)
└─ Alert at: $2,375 (95%)

ML Team: $1,000/month
├─ Training GPUs: $700
├─ Storage: $200
├─ Monitoring: $100
└─ Alert at: $950 (95%)

DevOps: $800/month
├─ Infrastructure: $500
├─ Monitoring: $200
├─ Backup: $100
└─ Alert at: $760 (95%)

Process:
├─ Monthly review of actual vs budget
├─ Quarterly planning for next quarter
├─ Approval needed for overages > 10%
└─ Root cause analysis for > 20% overage
```


## Cost Management Checklist

### Weekly

- [ ] Check for underutilized resources
- [ ] Monitor cost alerts
- [ ] Review new resources for tagging

### Monthly

- [ ] Generate cost report
- [ ] Analyze trends
- [ ] Update budget forecast
- [ ] Team cost review meeting

### Quarterly

- [ ] Review RI strategy
- [ ] Optimization opportunity review
- [ ] Budget planning for next quarter
- [ ] Vendor contract review

### Annually

- [ ] Cost forecast for year
- [ ] RI renewal decisions
- [ ] Architecture review for efficiency
- [ ] Benchmarking vs industry

---

**Next Resources:**

- See PERFORMANCE_OPTIMIZATION_GUIDE.ipynb for infrastructure tuning
- See DEPLOYMENT_AND_DEVOPS_GUIDE.ipynb for cost-aware deployment
